# Práctica guiada — De formulario a experiencia de inferencia

En S5 ya construisteis un formulario Streamlit que recoge los 11 atributos y llama a un gateway basado en el contrato de S4. En S6 **no se rediseña el contrato ni se vuelve a implementar el formulario**. El objetivo es convertir esa pantalla funcional en una interfaz operable: mantiene el resultado al haber un rerun, hace visible lo que está ocurriendo y enseña la información justa para interpretar e investigar una predicción.

Trabajad en parejas. Completad los `TODO`, ejecutad las comprobaciones y conservad el notebook como evidencia para el taller. La implementación queda para la Clase 2.

## Entrega de la práctica

Al terminar debéis poder enseñar cuatro decisiones concretas:

1. qué permanece en la sesión, qué se cachea y qué no se retiene;
2. la máquina `idle → loading → success/error`;
3. un plano de pantalla con componentes Streamlit concretos;
4. mensajes seguros para confianza baja, lentitud y error.

No se entrega código de inferencia nuevo. S4 sigue siendo la única frontera de validación y predicción.

## 1. Comprobar el punto de partida de S5

Ejecutad la celda. Debe demostrar que partís del formulario y gateway de S5, no de una segunda implementación. Después anotad una limitación que observaríais al cambiar un widget o al enviar una segunda predicción.

In [2]:
from pathlib import Path

repo_root = next(
    parent
    for parent in (Path.cwd(), *Path.cwd().parents)
    if (parent / 'semana5/modules/05-streamlit-basic-model-ui').is_dir()
)
s5_app = repo_root / 'semana5/modules/05-streamlit-basic-model-ui/solutions/02-first-streamlit/app.py'
source = s5_app.read_text(encoding='utf-8')

checks = {
    'reutiliza st.form': 'st.form(' in source,
    'recoge valores antes de inferir': 'collect_values' in source,
    'aún no usa session_state': 'st.session_state' not in source,
}
print('app de S5:', s5_app)
for label, passed in checks.items():
    print(f"{'✓' if passed else '✗'} {label}")
assert all(checks.values())

# escribid aquí una limitación observable de S5.
limitacion_s5 = 'S5 no conserva la última predicción en session_state; si hay un rerun sin enviar el formulario, el resultado desaparece y vuelve a mostrarse el aviso inicial.'
print('Limitación observada:', limitacion_s5)

app de S5: /home/gca21/master/mlops/muiaap-operacion-de-modelos/semana5/modules/05-streamlit-basic-model-ui/solutions/02-first-streamlit/app.py
✓ reutiliza st.form
✓ recoge valores antes de inferir
✓ aún no usa session_state
Limitación observada: S5 no conserva la última predicción en session_state; si hay un rerun sin enviar el formulario, el resultado desaparece y vuelve a mostrarse el aviso inicial.


## 2. Decidir dónde vive cada cosa

Un rerun vuelve a ejecutar el script. Una variable local desaparece; `st.session_state` conserva información de **esa sesión**; `st.cache_resource` reutiliza recursos como un gateway o modelo. Un recurso cacheado global puede compartirse entre usuarios, así que debe ser seguro para concurrencia.

Completad la tabla y justificad cada decisión. No guardéis secretos ni las 11 features en la telemetría. Los valores del último envío solo se conservan para permitir un reintento y no se muestran ni se registran.

| Clave o recurso | Dónde vive | ¿Sobrevive a un rerun? | ¿Se limpia al pulsar «Limpiar»? | Motivo |
| --- | --- | --- | --- | --- |
| `last_state` | `st.session_state` | Sí | Sí: vuelve a `idle` | Conserva fase, resultado presentable o error seguro para redibujar tras un rerun, no contiene las features originales. |
| `last_values` | `st.session_state` | Sí | Sí: se elimina para cerrar el reintento | Mantiene los valores del último envío solo para permitir reintentar, no se muestran ni se registran en telemetría. |
| `telemetry` | `st.session_state` (agregados) | Sí | No | Conserva contadores, latencias, errores y versiones. Limpiar la tarjeta no debe borrar la observabilidad de la sesión. |
| gateway / bundle | `st.cache_resource` | Sí, mientras viva la caché del proceso | No | Cargarlo una vez evita reconstruir el recurso en cada rerun, no guardar aquí datos de petición y considerar que puede compartirse entre sesiones. |
| respuesta de una petición | Variable local del controlador, su view model validado se copia a `last_state` | No de forma independiente, se vuelve a mostrar desde `last_state` | Sí, al limpiar `last_state` | La respuesta cruda solo se necesita durante la petición, se conserva únicamente la representación necesaria para mostrar el resultado sin retener el payload completo. |

Referencias: [`st.session_state`](https://docs.streamlit.io/develop/api-reference/caching-and-state/st.session_state) y [`st.cache_resource`](https://docs.streamlit.io/develop/api-reference/caching-and-state/st.cache_resource).

## 3. Diseñar la pantalla, no solo el estado

La interfaz debe tener cuatro zonas, en este orden:

1. **acción:** el formulario heredado de S5;
2. **operación:** qué está ocurriendo mientras se ejecuta;
3. **resultado:** categoría, confianza y latencia;
4. **trazabilidad:** versiones y `request_id`, visibles solo si se necesitan.

Elegid componentes que tengan una función. `st.columns` organiza, `st.status` comunica una operación, `st.metric` destaca un dato y `st.expander` aplaza el detalle técnico. No uséis una barra de progreso para la confianza: representa avance de una tarea, no certeza del modelo.

In [3]:
# Sustituid cada TODO por el componente y la justificación que usaríais.
# Las claves describen zonas de la interfaz; no son nuevos contratos de ML.

ui_plan = {
    'formulario': {'component': 'st.form', 'reason': 'Agrupa las 11 entradas y ejecuta la inferencia solo al enviar'},
    'estado_operacion': {'component': 'st.status', 'reason': 'Hace visible la fase loading mientras el gateway procesa la petición, sin confundirla con la confianza.'},
    'resumen_resultado': {'component': 'st.columns', 'reason': 'Ordena métricas de categoría, confianza y latencia para comparar el resultado'},
    'mensaje_interpretacion': {'component': 'st.warning', 'reason': 'Avisa de confianza baja o latencia alta sin presentar una salida válida como error.'},
    'trazabilidad': {'component': 'st.expander', 'reason': 'Deja versiones y request_id disponibles para investigar sin recargar la vista principal.'},
    'separacion_telemetria': {'component': 'st.divider', 'reason': 'Separa el resultado individual de los agregados de sesión.'},
}

allowed_components = {
    'st.form', 'st.status', 'st.empty', 'st.columns', 'st.container',
    'st.metric', 'st.success', 'st.info', 'st.warning', 'st.error',
    'st.expander', 'st.divider', 'st.caption',
}
required_zones = {
    'formulario', 'estado_operacion', 'resumen_resultado',
    'mensaje_interpretacion', 'trazabilidad', 'separacion_telemetria',
}

assert set(ui_plan) == required_zones
for zone, decision in ui_plan.items():
    assert set(decision) == {'component', 'reason'}, zone

pending = [zone for zone, decision in ui_plan.items() if 'TODO' in decision.values()]
invalid = [
    f"{zone}: {decision['component']}"
    for zone, decision in ui_plan.items()
    if 'TODO' not in decision['component'] and decision['component'] not in allowed_components
]
print('Zonas pendientes:', pending or 'ninguna')
print('Componentes no permitidos:', invalid or 'ninguno')
print('Comprobación final: pending == [] and invalid == []')

Zonas pendientes: ninguna
Componentes no permitidos: ninguno
Comprobación final: pending == [] and invalid == []


### Plano mínimo esperado

```text
Título y contexto breve

[ formulario de S5 en dos columnas ]

[ estado de la operación ]

[ Categoría ]  [ Confianza ]  [ Latencia ]
mensaje interpretativo de confianza y latencia
▶ Trazabilidad: versiones y request_id

────────────────────────────────────
Telemetría agregada de la sesión
```

Podéis entregar este plano en el notebook o redibujarlo en draw.io. Justificad cualquier cambio respecto a esta estructura.

## 4. Máquina de estados y recuperación

Completad la tabla. Una respuesta válida con latencia superior a 300 ms sigue en `success`: añade un aviso técnico, no se convierte en error. La confianza es una señal para revisar, no un diagnóstico ni una garantía.

| Estado | Evento de entrada | Zona de pantalla que cambia | Qué se muestra | Acción permitida | Evidencia |
| --- | --- | --- | --- | --- |
| `idle` | Inicio / limpiar | Panel de estado y resultado | Instrucción para completar el formulario, sin tarjeta de predicción | Introducir o editar datos y enviar | Fase `idle`: al limpiar se conserva la telemetría agregada. |
| `loading` | Enviar formulario | Panel de operación | Estado visible «Validando y ejecutando…»; se conserva el formulario | Esperar a que termine, no lanzar otra petición desde el mismo envío | Se emite `loading` antes de llamar al gateway y se mide la latencia. |
| `success` | Payload válido | Resultado y, si corresponde, aviso técnico | Categoría, confianza y latencia; versiones y `request_id` dentro de trazabilidad. Si supera 300 ms, aviso de lentitud sin cambiar a error | Revisar el resultado, modificar la entrada o reintentar | Payload validado, contadores de éxito y latencia. Fase permanece `success` aunque sea lento. |
| `error` | Excepción controlada | Panel de resultado/error | Mensaje accionable y `request_id`; nunca traceback, ruta, secreto ni payload completo | Corregir y reenviar, reintentar si es recuperable | Código de error y latencia agregados, sin guardar las features en telemetría. |

Recuperación: ante entrada inválida, indicar que revise los campos fuera de rango y permitir corregirlos y reenviar. Ante bundle ausente, informar de que el servicio no está disponible y ofrecer reintento o contactar al equipo responsable. Ante timeout, indicar que la petición tardó demasiado y permitir reintentar. En todos los casos no mostrar traceback, rutas internas, secretos ni el payload completo.

## 5. Escribir el copy de una interfaz responsable

Cada mensaje debe decir qué ha ocurrido, qué significa y qué puede hacer la persona. No debéis prometer que una predicción es correcta. Completad textos de una frase, sin terminología interna.

In [4]:
ux_copy = {
    'low_confidence': 'La confianza de esta predicción es baja.',
    'slow_success': 'La predicción está disponible, pero ha tardado más de lo esperado.',
    'invalid_input': 'Algunos valores no son válidos, revisa los campos indicados y reenvía el formulario.',
    'artifact_unavailable': 'El modelo no está disponible, inténtalo de nuevo más tarde o contacta con el equipo responsable.',
}

assert set(ux_copy) == {
    'low_confidence', 'slow_success', 'invalid_input', 'artifact_unavailable'
}
for scenario, message in ux_copy.items():
    assert isinstance(message, str) and message.strip(), scenario

pending_copy = [scenario for scenario, message in ux_copy.items() if 'TODO' in message]
print('Mensajes pendientes:', pending_copy or 'ninguno')
print('Revisión humana: ¿el texto evita certeza, traceback y culpabilizar a la persona?')

Mensajes pendientes: ninguno
Revisión humana: ¿el texto evita certeza, traceback y culpabilizar a la persona?


## 6. Preparar el taller

Entregad el notebook con los TODO completados. Antes de cerrar, otra pareja debe poder responder estas preguntas leyendo vuestro trabajo:

- ¿Qué se mantiene al pulsar otro widget y qué se elimina con «Limpiar último resultado»?
- ¿Qué función cargará el gateway una sola vez y cuál será su límite?
- ¿Qué componentes dibujará `render_state()` para `loading`, `success` y `error`?
- ¿Qué ve una persona con confianza baja o una respuesta lenta?
- ¿Dónde aparecen el `request_id` y las versiones sin invadir el resultado?

En el taller se implementarán estas decisiones en `session.py`, `policies.py`, `presentation.py`, `controller.py` y `app.py`.

## Debrief

- S5 no conserva la última predicción cuando ocurre un rerun.
- El gateway se puede reutilizar, el resultado pertenece a la sesión que hizo la petición.
- `st.status` muestra la carga. Métricas y mensajes muestran éxito o error.
- La categoría es la salida, la confianza indica incertidumbre y la latencia mide el tiempo.
- Se conservan el formulario, el contrato de entrada/salida y la presentación, cambia el transporte del gateway.